In [ ]:
import numpy as np
import sysl.symbolic as sls
import geolipi.symbolic as gls
from sysl.shader.evaluate import evaluate_to_shader
from sysl.shader_runtime.generate_shader_html import create_shader_html, make_jupyter_compatible_html
from IPython.display import display, HTML

settings = {
    "render_mode": "v1",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": False,
        "_AA": 1,
        "_RAYCAST_MAX_STEPS": 200,
    },
    "set_to_ubo": False,
    "export_params": False,
}



In [ ]:
# TODO: Test all combinators. 
primitive_1 = gls.Translate3D(gls.Cuboid3D((0.2,0.4, 0.8)), (-0.25, -0.5, -0.25,))
primitive_2 = gls.Translate3D(gls.Cuboid3D((0.2,0.4, 0.8,)), (0.25, 0.5, 0.25,))
prim_3 = gls.Translate3D(gls.Cuboid3D((0.25, 0.25, 0.25,)), (0.5, 0.5, 0.5))
prim_4 = gls.Translate3D(gls.Cuboid3D((0.125, 0.135, 0.115,)), (0.5, 0.5, 0.0))

test_exprs = [
    gls.Union(primitive_1, primitive_2),
    gls.Intersection(primitive_1, primitive_2),
    gls.Difference(primitive_1, primitive_2),
    gls.Intersection(gls.Complement(primitive_1), primitive_2),
    gls.SwitchedDifference(primitive_1, primitive_2),
    gls.SmoothUnion(primitive_1, primitive_2, 0.1),
    gls.SmoothIntersection(primitive_1, primitive_2, 0.1),
    gls.SmoothDifference(primitive_1, primitive_2, 0.1),
    gls.XOR(primitive_1, primitive_2),
    
    # gls.NarySmoothUnion(primitive_1, primitive_2, 0.1),
    # gls.NarySmoothIntersection(primitive_1, primitive_2, 0.1),
    # Translations, 
    gls.Union(primitive_1, 
        gls.Translate3D(primitive_2, (0.25, 0.0, 0.0,)),
        gls.Translate3D(primitive_2, (0.0, 0.35, 0.0,)),
        gls.Translate3D(primitive_2, (0.0, 0.0, 0.45,)),
        ),
    gls.Union(primitive_1, 
        gls.EulerRotate3D(primitive_2, (0.25, 0.0, 0.0,)),
        gls.EulerRotate3D(primitive_2, (0.0, 0.35, 0.0,)),
        gls.EulerRotate3D(primitive_2, (0.0, 0.0, 0.45,)),
        ),
    gls.Union(primitive_1, 
        gls.AxisAngleRotate3D(primitive_2, (0.25, 0.0, 0.0,)),
        gls.AxisAngleRotate3D(primitive_2, (0.0, 0.35, 0.0,)),
        gls.AxisAngleRotate3D(primitive_2, (0.0, 0.0, 0.45,)),
        ),
    gls.Union(
        gls.Scale3D(primitive_1, (1.2, 1.0, 1.0)),
        gls.Scale3D(primitive_1, (1.0, 1.4, 1.0)),
        gls.Scale3D(primitive_1, (1.0, 1.0, 1.8)),
    ),
    # Dilate Erode
    gls.Union(primitive_1, gls.Dilate3D(primitive_2, (0.1,))),
    gls.Union(primitive_1, gls.Erode3D(primitive_2, (0.1,))),
    gls.Union(primitive_1, gls.Onion3D(primitive_2, (0.1,))),
    gls.Union(primitive_1, gls.NegOnlyOnion3D(primitive_2, (0.1,))),
    
    gls.Union(primitive_1, gls.Distort3D(primitive_2, (0.1,))),
    gls.Union(primitive_1, gls.Twist3D(primitive_2, (0.5,))),
    gls.Union(primitive_1, gls.Bend3D(gls.Cuboid3D((0.2, 0.4, 0.8,)), (1.1,))),
    # Macros
    gls.ReflectCoords3D(primitive_2, (0.1, 0.2, 0.3)),
    gls.ReflectX3D(primitive_2,),
    gls.ReflectY3D(primitive_2,),
    gls.ReflectZ3D(primitive_2,),
    gls.TranslationSymmetry3D(prim_3, 
        (1, 0, 0), (1.0,), (5.0,)),
    gls.TranslationSymmetryX3D(prim_3, (1.2, ), (5.0,)),
    gls.TranslationSymmetryY3D(prim_3, (1.2, ), (5.0,)),
    gls.TranslationSymmetryZ3D(prim_3, (1.2, ), (7.0,)),
    gls.RotationSymmetryX3D(prim_4, (0.2, ), (5.0,)),
    gls.RotationSymmetryY3D(prim_4, (0.2, ), (5.0,)),
    gls.RotationSymmetryZ3D(prim_4, (0.2, ), (7.0,)),
    # SYSL ops:
    # GeomOnlySmoothUnion

]

# Test all Transforms. 
# Test Macros
# Test mix

In [ ]:
SEL_INDEX = 30
# 17 
cur_expr = test_exprs[SEL_INDEX]
print(cur_expr)
material = sls.MaterialV1((2.0,))
scene_with_material = sls.MatSolidV1(cur_expr, material)

# Get Shader Code
shader_code, uniforms, textures = evaluate_to_shader(scene_with_material, settings=settings)
# TO visualize in a browser:
with open("shader_code.glsl", "w") as f:
    f.write(shader_code)
html_code = create_shader_html(shader_code, uniforms, textures, show_controls=False)
# To visualize inline in jupyter notebook:
with open("test.html", "w") as f:
    f.write(html_code)
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))

In [ ]:
# Also check match between eval in render. 
from sysl.shader.utils.texture import recursive_encode_texture_tensor
from geolipi.torch_compute import Sketcher, recursive_evaluate
sketcher = Sketcher(n_dims=3, resolution=64)
secondary_sketcher = Sketcher(n_dims=2, resolution=128)

# cur_expr = gls.ArbitraryCappedCylinder3D((0.3, 0.0, 0.1, ), (0.8, 0.0, 0.7,), (0.1,))
eval_out = recursive_evaluate(cur_expr.tensor(), sketcher, secondary_sketcher)# [..., 0]
if len(eval_out.shape) == 2:
    eval_out = eval_out[..., 0]
expr = gls.SDFGrid3D(eval_out, "torch_eval", (0.2,))

expr = recursive_encode_texture_tensor(expr, sketcher)

new_expr = sls.MatSolidV1(expr, material)

final_scene = gls.Union(
    new_expr,
    gls.Translate3D(scene_with_material, (2.0, 0.0, 0.0))
)


shader_code, uniforms, textures = evaluate_to_shader(final_scene, settings=settings)
# TO visualize in a browser:
with open("shader_code.glsl", "w") as f:
    f.write(shader_code)
html_code = create_shader_html(shader_code, uniforms, textures, show_controls=False)
# To visualize inline in jupyter notebook:
with open("test.html", "w") as f:
    f.write(html_code)
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))
